# Tools

## Goal

Understanding:

- What a LangChain tool actually is, not just how to write `@tool`.
- Why does an LLM need tools.
- What makes a python function a LangChain tool?
- How does the LLM know which tools exist?
- Who decides when a tool should be called?
- Who actually executes the tool?
- How does the tool result get back into the LLM conversation?
- What is the difference between a Tool and simply calling a python function?
- How does this become the foundation for agents and LangGraph?

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

## Get Weather example tool

In [4]:
# Creating tool
from langchain.tools import tool

@tool
def get_weather(city: str):
    """Get the current weather for a city. """
    return f"The weather in {city} is sunny. "


**Conceptually**

```text
                LangChain Tool
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
      Name       Description    Schema
        │            │            │
 get_weather   Get current...   city: str
                     │
                     ↓
              Python function
```

In [5]:
print(get_weather.name) # Name of the function 
print(get_weather.description) # The description (what between the three quotes) 
print(get_weather.args) # the input to the Tool. here it is the return argument (city)

get_weather
Get the current weather for a city.
{'city': {'title': 'City', 'type': 'string'}}


In [6]:
# Creates tool object.
llm_with_tools = llm.bind_tools([get_weather]) 

1. `bind_tools()`: Tell the model what tools exist
2. `invoke()`: Ast the model what to do
3. tool execution: Actually run the python function.

In [ ]:
# Invoke the tool 
response = llm_with_tools.invoke( # Gemini call
    "What's the weather in Cairo"
)

In [8]:
print(response,"\n")
print(response.tool_calls)

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Cairo"}'}, '__gemini_function_call_thought_signatures__': {'call_3327829': 'EvQCCvECARFNMg+eJVwirHhLKZSuAMtZDV1ff/2YT43qareoxv+uplewD7iG/VeP6+piwxPUjgBtMn7ZWR/srRI5sLCpzoyARJgJlj30cSLT/lhsTXI8xDs/b5I9NHEVpt+dJzgWB9wnxBh0iyKZ2EWxdVaGF3dQQVdyt3GCp7RtwEhQlbHHx2PZvgFdMlznDHdjFIfqOykg7gxTkj+IvD8zfpiAvYzFAnTmVtrz3IMeDMe7UkLWmSD9bHTnfnWNU/2ZeXclYKlbf7qf+fJ/AMxLDAl//fXGnxJdxDHwnEoF9urq+CCpCgq8cKkiy1Q+FsrjK3//k0F38CsyxtAI+o17l4xt1L99jbCDDHJPEhs1bfsXIzw7QoPwP05j9p4Wo9tcUx9uaF9b1KRBLuh5sj+273Wo1SO5UBruDjSk3UVxebE0U3wNyZGihx5zRpFtC/f36iVCcgTEffo9ErSzDBkSEZbHaS7LKHoTWayGh6RSNkuMyxhC'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a033cd-6105-7d00-8801-ea28014c0518-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Cairo'}, 'id': 'call_3327829', 'type': 'tool_call'}] invalid_tool_calls=[] usage_meta

In [9]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Cairo'},
  'id': 'call_3327829',
  'type': 'tool_call'}]

In [10]:
tool_call  = response.tool_calls[0] 
tool_call 

{'name': 'get_weather',
 'args': {'city': 'Cairo'},
 'id': 'call_3327829',
 'type': 'tool_call'}

In [11]:
result = get_weather.invoke(tool_call['args'])
print(result)

The weather in Cairo is sunny. 


```text
Gemini
  │
  │ tool call
  ↓
AIMessage
  │
  │ response.tool_calls
  ↓
Application
  │
  │ get_weather.invoke(args)
  ↓
Tool
  │
  ↓
"the weather in Cairo is sunny"
```

## Tool message

sequence:

```text
HumanMessage
      ↓
AIMessage
      ↓
ToolMessage
      ↓
Gemini
```

In [5]:
from langchain_core.messages import ToolMessage

tool_call = response.tool_calls[0]

result = get_weather.invoke(tool_call["args"])

tool_message = ToolMessage(
    content=result,
    tool_call_id = tool_call["id"]
)

print(tool_message)

NameError: name 'response' is not defined

In [ ]:
# Build the human message.
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(content = "What's the weather in Cairo?"),
    response,
    tool_message,
]

```text
messages
│
├── HumanMessage
│      "What's the weather in Cairo?"
│
├── AIMessage
│      tool_calls:
│          get_weather(city="Cairo")
│
└── ToolMessage
       tool_call_id:
           call_1635569
       content:
           "The weather in Cairo is sunny."
```

In [14]:
final_response = llm_with_tools.invoke(messages)

In [15]:
print(final_response)
print(final_response.content)

content=[{'type': 'text', 'text': 'The weather in Cairo is currently sunny.', 'extras': {'signature': 'EmgKZgERTTIPC7MFBoo9OBzr6wokfdfHGfASMR1NuqqPtiX+JsX/9TTJf5u0+FHYlbPVoa2QgMgCj5f5Xk1A7Rb5m7+PDgjgMO3efttTJPEe2f14IHDNIn4Ed/WMbQsSMQ/afbJWARfdWA=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a033cd-6769-7d93-b5e2-1a2156b36aa4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 153, 'output_tokens': 8, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}
[{'type': 'text', 'text': 'The weather in Cairo is currently sunny.', 'extras': {'signature': 'EmgKZgERTTIPC7MFBoo9OBzr6wokfdfHGfASMR1NuqqPtiX+JsX/9TTJf5u0+FHYlbPVoa2QgMgCj5f5Xk1A7Rb5m7+PDgjgMO3efttTJPEe2f14IHDNIn4Ed/WMbQsSMQ/afbJWARfdWA=='}}]


Workflow:

```text
User
 │
 │ "What's the weather in Cairo?"
 ↓
llm_with_tools.invoke()
 │
 ↓
Gemini
 │
 │ AIMessage
 │ tool_calls = get_weather(Cairo)
 ↓
Application
 │
 │ get_weather.invoke({"city": "Cairo"})
 ↓
Tool
 │
 │ "The weather in Cairo is sunny."
 ↓
ToolMessage
 │
 │ tool_call_id = ...
 ↓
llm_with_tools.invoke(messages)
 │
 ↓
Gemini
 │
 │ AIMessage
 │ tool_calls = []
 │ content = "The weather in Cairo is currently sunny."
 ↓
User
```

## Database tool example

In [16]:
customers = {
    4821: {
        "name": "Ahmed",
        "balance": 12500,
        "status": "active",
    },
    7392: {
        "name": "Sara",
        "balance": 8200,
        "status": "active",
    },
}

In [17]:
# Normal function
def get_customer(customer_id: int):
    return customers.get(customer_id)

In [ ]:
# Create Tool

@tool
def get_customer(customer_id: int, field:str ):
    """Get customer information using the customer ID."""
    return customers[customer_id][field]

In [12]:
print(get_customer.name)
print(get_customer.description)
print(get_customer.args)

get_customer
Get customer information using the customer ID.
{'customer_id': {'title': 'Customer Id', 'type': 'integer'}, 'field': {'$ref': '#/$defs/CustomerField'}}


In [6]:
# Cerate field enum
from enum import Enum

class CustomerField (str, Enum):
    NAME = "name"
    BALANCE = "balance"
    STATUS = "status"

In [9]:
print(CustomerField.NAME)
print(CustomerField.BALANCE)
print(CustomerField.STATUS)

CustomerField.NAME
CustomerField.BALANCE
CustomerField.STATUS


In [24]:
# Create pydantic model

from pydantic import BaseModel

class CustomerQuery(BaseModel):
    customer_id : int
    field: CustomerField

In [25]:
CustomerQuery(
    customer_id = 4821,
    field = "balance"
)

CustomerQuery(customer_id=4821, field=<CustomerField.BALANCE: 'balance'>)

In [26]:
CustomerQuery(
    customer_id = 4821,
    field = "salary"
)

ValidationError: 1 validation error for CustomerQuery
field
  Input should be 'name', 'balance' or 'status' [type=enum, input_value='salary', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum

In [27]:
# Use pydantic schema with tool

@tool (args_schema = CustomerQuery)
def get_customer_field(query: CustomerQuery): # same as  def get_customer_field(customer_id: int, field: CustomerField) but without redundant the args
    """Get customer information using the customer ID."""
    return customers[query.customer_id][query.field]

In [28]:
print(get_customer_field.args
)

{'customer_id': {'title': 'Customer Id', 'type': 'integer'}, 'field': {'$ref': '#/$defs/CustomerField'}}


In [29]:
# Build model binding

llm_with_customer_tool = llm.bind_tools([get_customer_field])

In [30]:
response = llm_with_customer_tool.invoke(
    "what is the balance of customer 4821"
)

In [31]:

print(response.tool_calls)

[{'name': 'get_customer_field', 'args': {'customer_id': 4821, 'field': 'balance'}, 'id': 'call_430035', 'type': 'tool_call'}]
